# 1. Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType
from pyspark.sql.functions import col, trim, length, abs, when

# 2. Read Bronze Table

In [0]:
df = spark.table('workspace.bronze.crm_sales_details')

# 3. Silver Layer Transformations

## 3.1. Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

## 3.2. Cleaning Dates

In [0]:
def parse_yyyymmdd(c):
    return F.when(
        (F.col(c) == 0) | (F.length(F.col(c)) != 8),
        None
    ).otherwise(
        F.to_date(F.col(c).cast("string"), "yyyyMMdd")
    )

cols = ["sls_order_dt", "sls_ship_dt", "sls_due_dt"]

for c in cols:
    df = df.withColumn(c, parse_yyyymmdd(c))

In [0]:
df.limit(10).display()

## 3.3. Validate Order Ship and Due Date Sequence
Business Rule:
1. `order_date` should be NOT NULL
2. `order_date` ≤ `ship_date` ≤ `due_date`


In [0]:
# Data Validation : Checking date valid date sequence

df = df.withColumn(
    "valid_date_sequence",
    F.when(
        (col("sls_order_dt").isNotNull()) &
        (col("sls_order_dt") <= col("sls_ship_dt")) &
        (col("sls_ship_dt") <= col("sls_due_dt")),
        "Valid"
    ).otherwise("Invalid"
    )
)
df.limit(10).display()


In [0]:
# Data Quality Checks
df.filter(col('valid_date_sequence')=="Invalid").display()

## 3.4. Sales and Price Correction

In [0]:
df = (
    df
    .withColumn(
        "sls_sales",
        F.when(
            col("sls_sales").isNull() |
            (col("sls_sales") <= 0) |
            (col("sls_sales") != col("sls_quantity") * F.abs(col("sls_price"))),
            col("sls_quantity") * F.abs(col("sls_price"))
        ).otherwise(col("sls_sales"))
    )
    .withColumn(
        "sls_price",
        F.when(
            col("sls_price").isNull() | (col("sls_price") <= 0),
            F.when(
                col("sls_quantity") != 0,
                col("sls_sales") / col("sls_quantity")
            )
        ).otherwise(col("sls_price"))
    )
)

## 3.5. Renaming Columns

In [0]:
RENAME_MAP = {
    "sls_ord_num": "order_number",
    "sls_prd_key": "product_number",
    "sls_cust_id": "customer_id",
    "sls_order_dt": "order_date",
    "sls_ship_dt": "ship_date",
    "sls_due_dt": "due_date",
    "sls_sales": "sales_amount",
    "sls_quantity": "quantity",
    "sls_price": "price"
}

for old_name, new_name in RENAME_MAP.items():
  df = df.withColumnRenamed(old_name, new_name)

In [0]:
# Sanity Checks of Dataframe
df.limit(10).display()

# 4. Writing Table To Silver Layer

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_sales")

In [0]:
%sql
-- Sanity Checks of Silver Table
SELECT * FROM workspace.silver.crm_sales